# Lab 06 External V2 — 06 Alert Metrics

**Dataset:** Synthea Healthcare  
**Architecture:** External Delta tables  
**Compute:** Databricks Serverless compatible

## Purpose

Create the single-row `lab06_data_volume_metrics` external Delta table used by
the SQL Alert. The metric uses real monthly encounter history plus a controlled
low-volume simulated observation.

> Serverless compatibility rule: this notebook does **not** call
> `REFRESH TABLE`, `CACHE TABLE`, `UNCACHE TABLE`, or Spark cache-refresh APIs.


## 1. Runtime context

In [0]:
def ensure_text_widget(name: str, default: str, label: str) -> None:
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, default, label)


def ensure_dropdown_widget(
    name: str,
    default: str,
    choices: list[str],
    label: str,
) -> None:
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.dropdown(name, default, choices, label)


ensure_text_widget("catalog", "dbr_dev", "01 Catalog")
ensure_text_widget("source_schema", "parvinbadalov", "02 Source schema")
ensure_text_widget(
    "source_volume_name",
    "lab06_gold_analytics",
    "03 Source volume",
)
ensure_text_widget(
    "target_schema",
    "parvinbadalov_lab06_ext",
    "04 Target schema",
)
ensure_text_widget(
    "external_gold_root",
    "AUTO",
    "05 External Gold root",
)
ensure_dropdown_widget(
    "run_validation",
    "true",
    ["true", "false"],
    "06 Run validation",
)

catalog = dbutils.widgets.get("catalog").strip()
source_schema = dbutils.widgets.get("source_schema").strip()
source_volume_name = dbutils.widgets.get("source_volume_name").strip()
target_schema = dbutils.widgets.get("target_schema").strip()
external_gold_root = dbutils.widgets.get("external_gold_root").strip().rstrip("/")
run_validation = (
    dbutils.widgets.get("run_validation").strip().lower() == "true"
)

source_volume_path = (
    f"/Volumes/{catalog}/{source_schema}/{source_volume_name}"
)
source_csv_path = f"{source_volume_path}/source/csv"
reference_path = f"{source_volume_path}/reference"
target_schema_fqn = f"{catalog}.{target_schema}"

# Manual-run support:
# 05/06 are outside the recurring dev-runner chain, so AUTO derives the
# external Gold root from the already-created external dim_date table.
if (
    not external_gold_root
    or external_gold_root.upper() == "AUTO"
    or external_gold_root == "REPLACE_WITH_EXTERNAL_GOLD_ROOT"
):
    dim_date_table = f"{target_schema_fqn}.dim_date"

    if not spark.catalog.tableExists(dim_date_table):
        raise RuntimeError(
            "external_gold_root could not be auto-detected because "
            f"{dim_date_table} does not exist. Run 01_dimensions first, "
            "or enter the ABFSS external Gold root manually."
        )

    dim_detail = spark.sql(
        f"DESCRIBE DETAIL {dim_date_table}"
    ).first()

    dim_location = str(
        dim_detail.asDict().get("location", "")
    ).rstrip("/")

    suffix = "/dim_date"

    if not dim_location.lower().endswith(suffix):
        raise RuntimeError(
            "Could not derive external_gold_root from dim_date location: "
            f"{dim_location}"
        )

    external_gold_root = dim_location[:-len(suffix)]

if not external_gold_root.lower().startswith("abfss://"):
    raise ValueError(
        "external_gold_root must resolve to an abfss:// Azure storage path."
    )

print(f"Catalog            : {catalog}")
print(f"Source schema      : {source_schema}")
print(f"Source volume      : {source_volume_name}")
print(f"Target schema      : {target_schema}")
print(f"External Gold root : {external_gold_root}")
print(f"Run validation     : {run_validation}")
ensure_text_widget(
    "drop_threshold_pct",
    "30",
    "07 Alert drop threshold %",
)
ensure_text_widget(
    "simulated_observed_pct",
    "20",
    "08 Simulated observed % of baseline",
)

drop_threshold_pct = float(dbutils.widgets.get("drop_threshold_pct"))
simulated_observed_pct = float(dbutils.widgets.get("simulated_observed_pct"))

if not (0 < simulated_observed_pct < 100):
    raise ValueError("simulated_observed_pct must be between 0 and 100.")
if not (0 < drop_threshold_pct < 100):
    raise ValueError("drop_threshold_pct must be between 0 and 100.")

## 2. Build the alert metric

In [0]:
import sys
from pathlib import Path

from pyspark.sql import functions as F

current_dir = Path.cwd()
lab_root = current_dir.parent if current_dir.name == "notebooks" else current_dir

if str(lab_root) not in sys.path:
    sys.path.insert(0, str(lab_root))

from src.external_tables import (
    normalize_location,
    overwrite_external_delta,
    registered_table_location,
    register_external_delta_table,
    validate_registered_location,
)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_schema_fqn}")

print("Serverless-safe external-table helpers loaded.")

fact_table = f"{target_schema_fqn}.fact_encounters"
metrics_table = f"{target_schema_fqn}.lab06_data_volume_metrics"
metrics_location = f"{external_gold_root}/lab06_data_volume_metrics"

if not spark.catalog.tableExists(fact_table):
    raise RuntimeError(f"Missing fact table: {fact_table}")

monthly_counts = (
    spark.table(fact_table)
    .filter(F.col("encounter_date").isNotNull())
    .groupBy(F.trunc("encounter_date", "month").alias("encounter_month"))
    .agg(F.count("*").alias("encounter_count"))
    .orderBy("encounter_month")
)

month_rows = (
    monthly_counts
    .orderBy(F.desc("encounter_month"))
    .limit(4)
    .collect()
)

if len(month_rows) < 4:
    raise RuntimeError(
        "At least four observed months are required to build alert metrics."
    )

latest_month = month_rows[0]["encounter_month"]
baseline_rows = month_rows[1:4]
baseline_count = round(
    sum(int(r["encounter_count"]) for r in baseline_rows) / len(baseline_rows)
)

if baseline_count <= 0:
    raise RuntimeError("Computed baseline must be positive.")

observed_count = max(
    0,
    round(baseline_count * simulated_observed_pct / 100.0),
)

volume_drop_pct = round(
    (baseline_count - observed_count) / baseline_count * 100.0,
    2,
)

should_alert = 1 if volume_drop_pct >= drop_threshold_pct else 0
alert_status = "TRIGGERED" if should_alert == 1 else "OK"

metric_df = (
    spark.createDataFrame(
        [(
            int(should_alert),
            alert_status,
            "SIMULATED_LOW_VOLUME_FROM_REAL_BASELINE",
            latest_month,
            int(baseline_count),
            int(observed_count),
            float(volume_drop_pct),
            float(drop_threshold_pct),
        )],
        [
            "should_alert",
            "alert_status",
            "data_source",
            "test_month",
            "baseline_encounter_count",
            "observed_encounter_count",
            "volume_drop_pct",
            "drop_threshold_pct",
        ],
    )
    .withColumn("generated_at", F.current_timestamp())
)

overwrite_external_delta(
    spark,
    metric_df,
    metrics_table,
    metrics_location,
)

display(spark.table(metrics_table))

## 3. Validate

In [0]:
result = spark.table(metrics_table).first()

expected_drop = round(
    (
        result["baseline_encounter_count"]
        - result["observed_encounter_count"]
    )
    / result["baseline_encounter_count"]
    * 100.0,
    2,
)

checks = [
    ("baseline_positive", result["baseline_encounter_count"] > 0),
    ("observed_below_baseline",
     result["observed_encounter_count"] < result["baseline_encounter_count"]),
    ("drop_reconciles", float(result["volume_drop_pct"]) == float(expected_drop)),
    ("external_location",
     normalize_location(registered_table_location(spark, metrics_table))
     == normalize_location(metrics_location)),
]

display(spark.createDataFrame(
    [(n, "PASS" if ok else "FAIL") for n, ok in checks],
    ["validation_area","status"],
))

failed = [n for n, ok in checks if not ok]
if run_validation and failed:
    raise RuntimeError("06 Alert Metrics failed: " + ", ".join(failed))

print("LAB 06 EXTERNAL V2 — ALERT METRICS COMPLETE")
print(f"should_alert: {result['should_alert']}")
print(f"alert_status: {result['alert_status']}")
print("Serverless compatibility: PASS")
print("Unsupported cache-management commands: 0")